# Train SetFit-CEFR in a notebook

Clones [berstearns/setfit-cefr](https://github.com/berstearns/setfit-cefr), installs it, and runs `train.py` + `predict.py` end-to-end against a data folder you supply.

**One config governs the whole notebook.** Defaults live in `configs/notebook.yaml`; you override them by editing the `OVERRIDES` dict in cell 1 (and nothing else). Every subsequent cell reads from a single `cfg: NotebookConfig` object.

| Section          | Purpose                                                                     |
|------------------|-----------------------------------------------------------------------------|
| `cfg.input`      | Where your CSVs live, which filenames, which are train vs. test             |
| `cfg.output`     | Repo URL / clone dir, where models & predictions land                       |
| `cfg.train`      | YAML config path + CLI flags + dotted overrides forwarded to `train.py`     |
| `cfg.predict`    | Same, but for `predict.py`                                                  |

Expected CSVs (rename in `cfg.input.filenames` if yours differ):

| Role        | Filename                        |
|-------------|---------------------------------|
| training    | `norm-EFCAMDAT-train.csv`       |
| training    | `norm-EFCAMDAT-remainder.csv`   |
| external    | `norm-EFCAMDAT-test.csv`        |
| external    | `norm-KUPA-KEYS.csv`            |
| external    | `norm-CELVA-SP.csv`             |

Each CSV must have columns `writing_id,l1,cefr_level,text`.

## 1. `OVERRIDES` — the only cell you need to edit

Keys are dotted paths into `NotebookConfig`; values are the target Python values (lists, strings, ints…). Add or remove entries freely.

```python
OVERRIDES = {
    "input.data_dir": "/gdrive/MyDrive/cefr-csvs",
    "train.flag_args": ["--epochs", "2", "--seed", "7"],
    "train.dotted_overrides": ["training.max_length=256"],
    "output.models_root": "models",
}
```

In [ ]:
# ========== EDIT ME ==========================================================
OVERRIDES = {
    "input.data_dir": "/path/to/your/csvs",
}
# =============================================================================

# Bootstrap-only values — read *before* the package is installed, so they
# can't come from the validated `cfg` object yet. These default to the same
# values as `output.repo_url` / `output.repo_dir` in configs/notebook.yaml;
# if you override those above, they are picked up automatically.
BOOTSTRAP_REPO_URL = OVERRIDES.get(
    "output.repo_url", "https://github.com/berstearns/setfit-cefr.git"
)
BOOTSTRAP_REPO_DIR = OVERRIDES.get("output.repo_dir", "setfit-cefr")

## 2. Clone / update the repo

In [ ]:
import os
import subprocess
from pathlib import Path

in_repo = Path("pyproject.toml").exists() and Path("src/setfit_cefr").exists()

if in_repo:
    print("Already in the setfit-cefr repo root:", Path().resolve())
else:
    target = Path(BOOTSTRAP_REPO_DIR)
    if not target.exists():
        subprocess.check_call(["git", "clone", BOOTSTRAP_REPO_URL, str(target)])
    else:
        subprocess.check_call(["git", "-C", str(target), "pull", "--ff-only"])
    os.chdir(target)
    print("CWD ->", Path().resolve())

## 3. Install the package into this kernel

Editable install — takes ~2 min the first time on Colab.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

import setfit
import setfit_cefr

print("setfit      :", setfit.__version__)
print("setfit_cefr :", setfit_cefr.__version__)

## 4. Build the single `cfg` that governs everything

Layers `configs/notebook.yaml` (defaults) under `OVERRIDES` (your edits in cell 1) and validates the result. All subsequent cells reference `cfg`.

In [ ]:
from setfit_cefr import NotebookConfig

cfg = NotebookConfig.from_sources(
    yaml_path="configs/notebook.yaml",
    overrides=OVERRIDES,
)
print(cfg.to_yaml())

## 5. Link CSVs from `cfg.input.data_dir` into `cfg.output.local_data_dir`

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

src_dir = cfg.resolved_data_dir()
if not src_dir.is_dir():
    raise FileNotFoundError(
        f"cfg.input.data_dir does not exist: {src_dir}\n"
        f'Set OVERRIDES["input.data_dir"] to the correct path in cell 1.'
    )

Path(cfg.output.local_data_dir).mkdir(exist_ok=True)

pairs = cfg.source_file_pairs()
missing = [src for src, _ in pairs if not src.exists()]
if missing:
    raise FileNotFoundError(
        "Missing source CSV(s):\n  "
        + "\n  ".join(str(m) for m in missing)
        + '\nEdit OVERRIDES["input.filenames"] in cell 1 to match your filenames.'
    )

for src, dst in pairs:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src)
    except OSError:
        # Fallback for filesystems that don't support symlinks (e.g. Drive).
        shutil.copy2(src, dst)

REQUIRED = {"writing_id", "l1", "cefr_level", "text"}
for _, dst in pairs:
    head = pd.read_csv(dst, nrows=1)
    missing_cols = REQUIRED - set(head.columns)
    status = "OK" if not missing_cols else f"MISSING: {sorted(missing_cols)}"
    print(f"  {dst.name:<35}  cols={list(head.columns)}  {status}")

## 6. Preview the model hash (dry run)

Runs `train.py --dry-run` with the flags + overrides from `cfg.train`. Confirms the config is valid and prints the content-addressed model hash before you commit to a real training run.

In [ ]:
import subprocess
import sys
from pathlib import Path


def _build_cli(run_cfg, extra: list[str]) -> list[str]:
    """Assemble argv for train.py / predict.py from a RunConfig."""
    argv: list[str] = []
    if run_cfg.config_path is not None:
        argv += ["--config", run_cfg.config_path]
    argv += list(run_cfg.flag_args)
    for dotted in run_cfg.dotted_overrides:
        argv += ["--override", dotted]
    argv += extra
    return argv


dry = subprocess.check_output(
    [sys.executable, "train.py", *_build_cli(cfg.train, ["--dry-run"])],
    text=True,
)
model_hash = dry.strip().splitlines()[-1]
print("Model hash preview:", model_hash)
print("Would write to    :", Path(cfg.output.models_root) / model_hash)

## 7. Train

Streams `train.py` output live; captures the output folder from the final stdout line.

In [ ]:
import subprocess
import sys

proc = subprocess.Popen(
    [sys.executable, "train.py", *_build_cli(cfg.train, [])],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
last_line = ""
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    if line.strip():
        last_line = line.strip()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"train.py exited {proc.returncode}")

model_dir = last_line
print("\n--> model_dir =", model_dir)

## 8. Predict on the external test sets

Feeds `cfg.local_test_files()` to `predict.py`. Anything you set in `cfg.predict` (extra flags, dotted overrides) is forwarded.

In [ ]:
import subprocess
import sys

test_files = [str(p) for p in cfg.local_test_files()]
extra = ["--model", model_dir, "--test-files", *test_files]

proc = subprocess.Popen(
    [sys.executable, "predict.py", *_build_cli(cfg.predict, extra)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
last_line = ""
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
    if line.strip():
        last_line = line.strip()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"predict.py exited {proc.returncode}")

pred_dir = last_line
print("\n--> pred_dir =", pred_dir)

## 9. Inspect artefacts

In [ ]:
from pathlib import Path

print("Model folder:", model_dir)
for p in sorted(Path(model_dir).iterdir()):
    print("  ", p.name)

print("\nPrediction folder:", pred_dir)
for p in sorted(Path(pred_dir).iterdir()):
    print("  ", p.name)

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

report_md = Path(pred_dir) / "report.md"
display(Markdown(report_md.read_text(encoding="utf-8")))

### Next steps — all controlled from cell 1's `OVERRIDES`

- **Swap the backbone**
  ```python
  OVERRIDES["train.flag_args"] = [
      "--model-name", "sentence-transformers/all-mpnet-base-v2"
  ]
  ```
- **Seed sweep** — `"train.flag_args": ["--seed", "1"]`, `["--seed", "2"]`, …
- **Larger few-shot** — `"train.flag_args": ["--sample-per-class", "128"]`.
- **Skip training, just predict** — comment out cells 6–7 and set `model_dir = "models/<existing-hash>"`; cell 8 still works.
- **Rename CSVs** — `OVERRIDES["input.filenames"] = {"train": "my-train.csv", ...}`; keys in `train_keys`/`test_keys` must match.